# Create Validation and Test Label Files

## Temporal split

- Training stages: 2010–2023
- Validation: January 1–December 30, 2024
- Test: January 1, 2025–July 31, 2026

## Boundary purge

Because every image predicts the following 24 hours:

- December 31, 2023 is removed from Stage 3 training.
- December 31, 2024 is removed from validation.

This prevents prediction windows from crossing into the following dataset.

In [1]:
from pathlib import Path
import shutil

import pandas as pd

# Find the repository root.
possible_roots = [Path.cwd(), *Path.cwd().parents]

ROOT = next(
    path for path in possible_roots
    if (path / "data_labeling").is_dir()
)

LABEL_ROOT = (
    ROOT
    / "data_labeling"
    / "data_labels"
)

SIMPLIFIED_LABEL_FILE = (
    LABEL_ROOT
    / "simplified_data_labels"
    / "labels_2019_2026_july_binary.csv"
)

STAGE_DIRECTORY = (
    LABEL_ROOT
    / "continual_stages_2010_2023_candidate_A"
)

STAGE3_TRAIN_FILE = (
    STAGE_DIRECTORY
    / "Stage3_train.csv"
)

# Backup allows the notebook to be rerun safely.
STAGE3_BACKUP_FILE = (
    STAGE_DIRECTORY
    / "Stage3_train_before_2024_boundary_purge.csv"
)

EVALUATION_DIRECTORY = (
    LABEL_ROOT
    / "future_evaluation_labels"
)

VALIDATION_FILE = (
    EVALUATION_DIRECTORY
    / "validation_2024.csv"
)

TEST_FILE = (
    EVALUATION_DIRECTORY
    / "test_2025_2026_july.csv"
)

print("Repository:", ROOT)
print("Stage 3 training file:", STAGE3_TRAIN_FILE)
print("Validation output:", VALIDATION_FILE)
print("Test output:", TEST_FILE)

assert SIMPLIFIED_LABEL_FILE.is_file()
assert STAGE3_TRAIN_FILE.is_file()

Repository: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-
Stage 3 training file: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2023_candidate_A/Stage3_train.csv
Validation output: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/future_evaluation_labels/validation_2024.csv
Test output: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/future_evaluation_labels/test_2025_2026_july.csv


In [2]:
def add_timestamp_columns(data):
    """
    Extract timestamp and year from each magnetogram path.
    """

    result = data.copy()

    timestamp_parts = result["label"].str.extract(
        r"HMI\.m(?P<date>\d{4}\.\d{2}\.\d{2})_"
        r"(?P<time>\d{2}\.\d{2}\.\d{2})"
    )

    result["timestamp"] = pd.to_datetime(
        timestamp_parts["date"]
        + " "
        + timestamp_parts["time"],
        format="%Y.%m.%d %H.%M.%S",
        errors="coerce",
    )

    result["year"] = result["timestamp"].dt.year

    if result["timestamp"].isna().any():
        raise ValueError(
            "Some timestamps could not be extracted."
        )

    return result

In [3]:
# If a backup already exists, use it as the original source.
# This makes the notebook safe to rerun.
stage3_source_file = (
    STAGE3_BACKUP_FILE
    if STAGE3_BACKUP_FILE.exists()
    else STAGE3_TRAIN_FILE
)

stage3_original = pd.read_csv(
    stage3_source_file
)

stage3_original = add_timestamp_columns(
    stage3_original
)

stage3_boundary_mask = (
    stage3_original["timestamp"].dt.date
    == pd.Timestamp("2023-12-31").date()
)

removed_stage3_boundary = stage3_original[
    stage3_boundary_mask
].copy()

stage3_corrected = stage3_original[
    ~stage3_boundary_mask
].copy()

print("Original Stage 3 training rows:", len(stage3_original))
print("Removed boundary rows:", len(removed_stage3_boundary))
print(
    "Removed FL:",
    int(removed_stage3_boundary["goes_class"].sum()),
)
print(
    "Removed NF:",
    len(removed_stage3_boundary)
    - int(removed_stage3_boundary["goes_class"].sum()),
)
print("Corrected Stage 3 rows:", len(stage3_corrected))

display(
    removed_stage3_boundary[
        ["label", "goes_class", "timestamp"]
    ].head()
)

Original Stage 3 training rows: 7319
Removed boundary rows: 24
Removed FL: 24
Removed NF: 0
Corrected Stage 3 rows: 7295


,label,goes_class,timestamp
7295,2023/12/31/HMI.m2023.12.31_00.00.00.jpg,1,2023-12-31 00:00:00
7296,2023/12/31/HMI.m2023.12.31_01.00.00.jpg,1,2023-12-31 01:00:00
7297,2023/12/31/HMI.m2023.12.31_02.00.00.jpg,1,2023-12-31 02:00:00
7298,2023/12/31/HMI.m2023.12.31_03.00.00.jpg,1,2023-12-31 03:00:00
7299,2023/12/31/HMI.m2023.12.31_04.00.00.jpg,1,2023-12-31 04:00:00


In [4]:
future_labels = pd.read_csv(
    SIMPLIFIED_LABEL_FILE
)

future_labels = add_timestamp_columns(
    future_labels
)

# Validation starts January 1, 2024 and ends
# before December 31, 2024.
validation_mask = (
    (future_labels["timestamp"] >= "2024-01-01 00:00:00")
    & (future_labels["timestamp"] < "2024-12-31 00:00:00")
)

validation_data = future_labels[
    validation_mask
].copy()

# Inspect the validation boundary images being excluded.
removed_validation_boundary = future_labels[
    (
        future_labels["timestamp"].dt.date
        == pd.Timestamp("2024-12-31").date()
    )
].copy()

# Test starts January 1, 2025 and includes all available
# samples through July 2026.
test_mask = (
    future_labels["timestamp"]
    >= "2025-01-01 00:00:00"
)

test_data = future_labels[
    test_mask
].copy()

print("Validation rows:", len(validation_data))
print(
    "Validation boundary rows removed:",
    len(removed_validation_boundary),
)
print(
    "Removed validation FL:",
    int(removed_validation_boundary["goes_class"].sum()),
)
print("Test rows:", len(test_data))
print("Test ends:", test_data["timestamp"].max())

Validation rows: 8611
Validation boundary rows removed: 24
Removed validation FL: 24
Test rows: 13365
Test ends: 2026-07-31 23:00:00


In [5]:
def summarize_dataset(name, data):
    total = len(data)
    fl_count = int(data["goes_class"].sum())
    nf_count = total - fl_count

    return {
        "Dataset": name,
        "Total": total,
        "NF": nf_count,
        "FL": fl_count,
        "FL percent": (
            100 * fl_count / total
            if total > 0
            else 0
        ),
        "NF:FL": (
            nf_count / fl_count
            if fl_count > 0
            else float("inf")
        ),
        "Start": data["timestamp"].min(),
        "End": data["timestamp"].max(),
    }


distribution_summary = pd.DataFrame([
    summarize_dataset(
        "Corrected Stage 3 train",
        stage3_corrected,
    ),
    summarize_dataset(
        "2024 validation",
        validation_data,
    ),
    summarize_dataset(
        "2025–July 2026 test",
        test_data,
    ),
])

display(
    distribution_summary.round({
        "FL percent": 2,
        "NF:FL": 2,
    })
)

,Dataset,Total,NF,FL,FL percent,NF:FL,Start,End
0,Corrected Stage 3 train,7295,3985,3310,45.37,1.20,2023-02-26 20:00:00,2023-12-30 23:00:00
1,2024 validation,8611,4321,4290,49.82,1.01,2024-01-01 00:00:00,2024-12-30 23:00:00
2,2025–July 2026 test,13365,8620,4745,35.50,1.82,2025-01-01 00:00:00,2026-07-31 23:00:00


Verify the temporal gaps

In [6]:
stage3_last_time = stage3_corrected[
    "timestamp"
].max()

validation_first_time = validation_data[
    "timestamp"
].min()

validation_last_time = validation_data[
    "timestamp"
].max()

test_first_time = test_data[
    "timestamp"
].min()

train_validation_difference = (
    validation_first_time
    - stage3_last_time
)

validation_test_difference = (
    test_first_time
    - validation_last_time
)

print("Last Stage 3 training image:", stage3_last_time)
print("First validation image:", validation_first_time)
print(
    "Time between training and validation:",
    train_validation_difference,
)

print()

print("Last validation image:", validation_last_time)
print("First test image:", test_first_time)
print(
    "Time between validation and test:",
    validation_test_difference,
)

assert train_validation_difference >= pd.Timedelta(
    hours=25
)

assert validation_test_difference >= pd.Timedelta(
    hours=25
)

print("\nBoth temporal boundary checks passed.")

Last Stage 3 training image: 2023-12-30 23:00:00
First validation image: 2024-01-01 00:00:00
Time between training and validation: 1 days 01:00:00

Last validation image: 2024-12-30 23:00:00
First test image: 2025-01-01 00:00:00
Time between validation and test: 1 days 01:00:00

Both temporal boundary checks passed.


In [7]:
datasets_to_check = {
    "Corrected Stage 3": stage3_corrected,
    "Validation": validation_data,
    "Test": test_data,
}

for name, data in datasets_to_check.items():
    assert data["label"].notna().all()
    assert data["goes_class"].isin([0, 1]).all()
    assert not data["label"].duplicated().any()

    print(
        f"{name}: "
        f"rows={len(data):,}, "
        f"duplicates=0, "
        f"missing=0"
    )

Corrected Stage 3: rows=7,295, duplicates=0, missing=0
Validation: rows=8,611, duplicates=0, missing=0
Test: rows=13,365, duplicates=0, missing=0


Verify that no datasets overlap

In [8]:
stage_paths = set()

for stage_number in [1, 2, 3]:
    for split_name in ["train", "holdout"]:
        file_path = (
            STAGE_DIRECTORY
            / f"Stage{stage_number}_{split_name}.csv"
        )

        stage_file = pd.read_csv(file_path)

        # Use the corrected Stage 3 training data instead
        # of the current saved version.
        if stage_number == 3 and split_name == "train":
            paths = set(stage3_corrected["label"])
        else:
            paths = set(stage_file["label"])

        overlap = stage_paths.intersection(paths)

        assert not overlap, (
            f"Overlap found in {file_path.name}"
        )

        stage_paths.update(paths)

validation_paths = set(validation_data["label"])
test_paths = set(test_data["label"])

assert stage_paths.isdisjoint(validation_paths)
assert stage_paths.isdisjoint(test_paths)
assert validation_paths.isdisjoint(test_paths)

print("No overlap between training stages and validation.")
print("No overlap between training stages and test.")
print("No overlap between validation and test.")

No overlap between training stages and validation.
No overlap between training stages and test.
No overlap between validation and test.


In [9]:
stage3_output = (
    stage3_corrected[
        ["label", "goes_class"]
    ]
    .sort_values("label")
    .reset_index(drop=True)
)

validation_output = (
    validation_data[
        ["label", "goes_class"]
    ]
    .sort_values("label")
    .reset_index(drop=True)
)

test_output = (
    test_data[
        ["label", "goes_class"]
    ]
    .sort_values("label")
    .reset_index(drop=True)
)

stage3_output["goes_class"] = (
    stage3_output["goes_class"].astype(int)
)

validation_output["goes_class"] = (
    validation_output["goes_class"].astype(int)
)

test_output["goes_class"] = (
    test_output["goes_class"].astype(int)
)

print("Corrected Stage 3 preview:")
display(stage3_output.head())

print("Validation preview:")
display(validation_output.head())

print("Test preview:")
display(test_output.head())

Corrected Stage 3 preview:


,label,goes_class
0,2023/02/26/HMI.m2023.02.26_20.00.00.jpg,0
1,2023/02/26/HMI.m2023.02.26_21.00.00.jpg,0
2,2023/02/26/HMI.m2023.02.26_22.00.00.jpg,0
3,2023/02/26/HMI.m2023.02.26_23.00.00.jpg,0
4,2023/02/27/HMI.m2023.02.27_00.00.00.jpg,0


Validation preview:


,label,goes_class
0,2024/01/01/HMI.m2024.01.01_00.00.00.jpg,1
1,2024/01/01/HMI.m2024.01.01_01.00.00.jpg,1
2,2024/01/01/HMI.m2024.01.01_02.00.00.jpg,1
3,2024/01/01/HMI.m2024.01.01_03.00.00.jpg,1
4,2024/01/01/HMI.m2024.01.01_04.00.00.jpg,1


Test preview:


,label,goes_class
0,2025/01/01/HMI.m2025.01.01_00.00.00.jpg,1
1,2025/01/01/HMI.m2025.01.01_01.00.00.jpg,1
2,2025/01/01/HMI.m2025.01.01_02.00.00.jpg,1
3,2025/01/01/HMI.m2025.01.01_03.00.00.jpg,1
4,2025/01/01/HMI.m2025.01.01_04.00.00.jpg,1


In [10]:
assert len(removed_stage3_boundary) == 24
assert removed_stage3_boundary["goes_class"].sum() == 24

assert len(removed_validation_boundary) == 24
assert removed_validation_boundary["goes_class"].sum() == 24

assert len(stage3_output) == len(stage3_original) - 24

assert validation_output["goes_class"].nunique() == 2
assert test_output["goes_class"].nunique() == 2

print("Final checks passed.")
print("The files are ready to save.")

Final checks passed.
The files are ready to save.


In [11]:
ALLOW_OVERWRITE_EVALUATION_FILES = False

# Check output conflicts before writing anything.
for output_file in [VALIDATION_FILE, TEST_FILE]:
    if (
        output_file.exists()
        and not ALLOW_OVERWRITE_EVALUATION_FILES
    ):
        raise FileExistsError(
            f"{output_file} already exists. "
            "Set ALLOW_OVERWRITE_EVALUATION_FILES=True "
            "only if you intend to replace it."
        )

EVALUATION_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

# Back up the original Stage 3 file once.
if not STAGE3_BACKUP_FILE.exists():
    shutil.copy2(
        STAGE3_TRAIN_FILE,
        STAGE3_BACKUP_FILE,
    )

# Replace Stage 3 training with the leakage-safe version.
stage3_output.to_csv(
    STAGE3_TRAIN_FILE,
    index=False,
)

# Save future evaluation labels.
validation_output.to_csv(
    VALIDATION_FILE,
    index=False,
)

test_output.to_csv(
    TEST_FILE,
    index=False,
)

print("Saved corrected Stage 3 training file:")
print(STAGE3_TRAIN_FILE)

print("\nSaved backup:")
print(STAGE3_BACKUP_FILE)

print("\nSaved validation labels:")
print(VALIDATION_FILE)

print("\nSaved test labels:")
print(TEST_FILE)

Saved corrected Stage 3 training file:
/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2023_candidate_A/Stage3_train.csv

Saved backup:
/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2023_candidate_A/Stage3_train_before_2024_boundary_purge.csv

Saved validation labels:
/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/future_evaluation_labels/validation_2024.csv

Saved test labels:
/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/future_evaluation_labels/test_2025_2026_july.csv


In [12]:
saved_files = {
    "Stage 3 train": STAGE3_TRAIN_FILE,
    "2024 validation": VALIDATION_FILE,
    "2025–2026 test": TEST_FILE,
}

verification_rows = []

for name, file_path in saved_files.items():
    saved = pd.read_csv(file_path)

    assert list(saved.columns) == [
        "label",
        "goes_class",
    ]

    assert saved["goes_class"].isin([0, 1]).all()
    assert not saved["label"].duplicated().any()
    assert not saved.isna().any().any()

    verification_rows.append({
        "Dataset": name,
        "File": file_path.name,
        "Rows": len(saved),
        "NF": int(
            (saved["goes_class"] == 0).sum()
        ),
        "FL": int(
            (saved["goes_class"] == 1).sum()
        ),
        "Duplicates": int(
            saved["label"].duplicated().sum()
        ),
        "Missing": int(
            saved.isna().sum().sum()
        ),
    })

verification = pd.DataFrame(
    verification_rows
)

display(verification)

print("All saved files passed verification.")

,Dataset,File,Rows,NF,FL,Duplicates,Missing
0,Stage 3 train,Stage3_train.csv,7295,3985,3310,0,0
1,2024 validation,validation_2024.csv,8611,4321,4290,0,0
2,2025–2026 test,test_2025_2026_july.csv,13365,8620,4745,0,0


All saved files passed verification.
